# 06. Bank failure records

Clean and check the FDIC failure records. Exclude assistance events and keep one record per genuine failure. Notebook 07 constructs the target.


## 1. Load configuration and verify the source contract

In [ ]:
# Load configuration and verify the source contract

from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import re
from datetime import date, datetime
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

PROJECT_ROOT = Path(
    os.environ.get(
        "DISSERTATION_PROJECT_ROOT",
        os.environ.get("DATA_FINAL_ROOT", "/content/drive/MyDrive/Data Final"),
    )
).expanduser().resolve()

# Accept the dedicated main analysis configuration when present, while remaining compatible
# with the existing shared config.py used by the executed Phase 1 notebooks.
explicit_config = os.environ.get("V2_CONFIG_PATH")
config_candidates = [
    Path(explicit_config).expanduser() if explicit_config else None,
    PROJECT_ROOT / "config_v2.py",
    PROJECT_ROOT / "config.py",
    Path.cwd() / "config_v2.py",
    Path.cwd() / "config.py",
]
config_matches = []
for candidate in config_candidates:
    if candidate is not None and candidate.is_file():
        resolved = candidate.resolve()
        if resolved not in config_matches:
            config_matches.append(resolved)
if not config_matches:
    raise FileNotFoundError(
        "No shared configuration was found. Expected config_v2.py or config.py "
        f"under {PROJECT_ROOT}, or set V2_CONFIG_PATH explicitly."
    )
CONFIG_PATH = config_matches[0]
config_spec = importlib.util.spec_from_file_location("dissertation_config_v2", CONFIG_PATH)
config_v2 = importlib.util.module_from_spec(config_spec)
config_spec.loader.exec_module(config_v2)

ANALYSIS_VERSION = getattr(config_v2, "ANALYSIS_VERSION", "v2_bank_lag1_main")
ELIGIBLE_BANK_CLASSES = tuple(getattr(config_v2, "ELIGIBLE_BANK_CLASSES", ("N", "NM", "SM")))
TARGET_HORIZON_DAYS = int(getattr(config_v2, "TARGET_HORIZON_DAYS", 365))
EXPECTED_ASSISTANCE_EVENTS = int(getattr(config_v2, "EXPECTED_ASSISTANCE_EVENTS", 10))
EXPECTED_EARLIEST_FAILURE = getattr(config_v2, "EXPECTED_EARLIEST_FAILURE", "2001-02-02")
EXPECTED_EXCEL_DATE_CELLS = int(getattr(config_v2, "EXPECTED_EXCEL_DATE_CELLS", 157))
EXPECTED_FDIC_SOURCE_ROWS = int(getattr(config_v2, "EXPECTED_FDIC_SOURCE_ROWS", 497))
EXPECTED_GENUINE_FAILURES = int(getattr(config_v2, "EXPECTED_GENUINE_FAILURES", 487))
EXPECTED_LATEST_FAILURE = getattr(config_v2, "EXPECTED_LATEST_FAILURE", "2026-05-01")
EXPECTED_TEXT_FAILURE_DATES = int(getattr(config_v2, "EXPECTED_TEXT_FAILURE_DATES", 340))
FINAL_TEST_END = getattr(config_v2, "FINAL_TEST_END", "2025-06-30")
OUTCOME_OBSERVED_THROUGH = getattr(config_v2, "OUTCOME_OBSERVED_THROUGH", "2026-07-20")
EXPECTED_GENUINE_FAILURE_COUNTS_BY_YEAR = getattr(
    config_v2,
    "EXPECTED_GENUINE_FAILURE_COUNTS_BY_YEAR",
    {
        2001: 3, 2002: 10, 2003: 2, 2004: 3, 2005: 0, 2006: 0,
        2007: 1, 2008: 19, 2009: 120, 2010: 132, 2011: 84,
        2012: 40, 2013: 23, 2014: 14, 2015: 8, 2016: 5, 2017: 6,
        2018: 0, 2019: 3, 2020: 4, 2021: 0, 2022: 0, 2023: 5,
        2024: 2, 2025: 1, 2026: 2,
    },
)
OUTPUT_ROOT = Path(
    os.environ.get(
        "NB06_OUTPUT_ROOT",
        PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION / "06_FDIC_Failure_Data_Cleaning",
    )
).expanduser().resolve()

def resolve_fdic_workbook() -> Path:
    explicit_input = os.environ.get("FDIC_RAW_WORKBOOK")
    if explicit_input:
        path = Path(explicit_input).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f"FDIC_RAW_WORKBOOK does not exist: {path}")
        return path
    candidates = [
        PROJECT_ROOT / "Raw Data" / "FDIC" / "FDIC Data.xlsx",
        PROJECT_ROOT / "data" / "raw" / "FDIC Data.xlsx",
        PROJECT_ROOT / "FDIC Data.xlsx",
    ]
    matches = [path.resolve() for path in candidates if path.is_file()]
    if len(matches) != 1:
        raise FileNotFoundError(
            "Expected exactly one FDIC Data.xlsx. Set FDIC_RAW_WORKBOOK explicitly. "
            f"Detected: {matches}"
        )
    return matches[0]

INPUT_WORKBOOK = resolve_fdic_workbook()
OUTPUT_DIR = OUTPUT_ROOT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALLOW_CSV_ONLY = os.environ.get("NB06_ALLOW_CSV_ONLY", "0") == "1"
print("Shared configuration:", CONFIG_PATH)

REQUIRED_RAW_COLUMNS = [
    "CERT", "CHCLASS1", "CITYST", "COST", "FAILDATE", "FIN", "ID",
    "NAME", "QBFASSET", "QBFDEP", "RESTYPE", "RESTYPE1", "SAVR",
]

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

xls = pd.ExcelFile(INPUT_WORKBOOK)
assert "FDIC_Raw" in xls.sheet_names, "Required source sheet FDIC_Raw is missing."
raw = pd.read_excel(INPUT_WORKBOOK, sheet_name="FDIC_Raw")

schema_audit = pd.DataFrame({
    "column_position": range(1, len(raw.columns) + 1),
    "column_name": list(raw.columns),
    "pandas_dtype": [str(dtype) for dtype in raw.dtypes],
    "nonmissing_observations": [int(raw[column].notna().sum()) for column in raw.columns],
    "missing_observations": [int(raw[column].isna().sum()) for column in raw.columns],
})
required_column_audit = pd.DataFrame({
    "required_column": REQUIRED_RAW_COLUMNS,
    "present": [column in raw.columns for column in REQUIRED_RAW_COLUMNS],
})
unexpected_raw_columns = sorted(set(raw.columns) - set(REQUIRED_RAW_COLUMNS))
assert required_column_audit["present"].all(), required_column_audit.loc[~required_column_audit["present"]]
assert len(raw) == EXPECTED_FDIC_SOURCE_ROWS, f"Expected 497 rows, found {len(raw):,}."
assert not raw.columns.duplicated().any(), "Duplicate source-column names detected."
assert raw.shape[1] == 13 and not unexpected_raw_columns

input_summary = pd.DataFrame([
    {"metric": "analysis_version", "value": ANALYSIS_VERSION},
    {"metric": "input_workbook", "value": INPUT_WORKBOOK.name},
    {"metric": "input_sha256", "value": sha256_file(INPUT_WORKBOOK)},
    {"metric": "source_sheet", "value": "FDIC_Raw"},
    {"metric": "source_rows", "value": len(raw)},
    {"metric": "source_columns", "value": raw.shape[1]},
    {"metric": "target_horizon_days", "value": TARGET_HORIZON_DAYS},
    {"metric": "eligible_bank_classes", "value": ",".join(ELIGIBLE_BANK_CLASSES)},
])
input_gate = pd.DataFrame([
    {"check": "source_rows_equal_497", "passed": len(raw) == EXPECTED_FDIC_SOURCE_ROWS},
    {"check": "source_columns_equal_13", "passed": raw.shape[1] == 13},
    {"check": "all_required_columns_present", "passed": required_column_audit["present"].all()},
    {"check": "no_unexpected_columns", "passed": not unexpected_raw_columns},
    {"check": "target_horizon_equal_365", "passed": TARGET_HORIZON_DAYS == 365},
    {"check": "bank_scope_equal_N_NM_SM", "passed": ELIGIBLE_BANK_CLASSES == ("N", "NM", "SM")},
])
input_gate["status"] = np.where(input_gate["passed"], "PASS", "FAIL")
assert input_gate["passed"].all(), input_gate.loc[~input_gate["passed"]]

input_summary.to_csv(OUTPUT_DIR / "cell_01_input_summary.csv", index=False)
schema_audit.to_csv(OUTPUT_DIR / "cell_01_input_schema.csv", index=False)
input_gate.to_csv(OUTPUT_DIR / "cell_01_input_validation_gate.csv", index=False)

print(input_summary.to_string(index=False))
print("\nCell 01 status: PASS")

## 2. Standardise and independently reconcile mixed-format failure dates

In [ ]:
# Standardise and independently reconcile mixed-format failure dates

def raw_date_as_text(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (pd.Timestamp, datetime, date, np.datetime64)):
        return pd.Timestamp(value).strftime("%Y-%m-%d")
    return str(value).strip()


def parse_mixed_fdic_failure_dates(series: pd.Series):
    # Excel date-formatted cells were saved after UK-locale interpretation of US
    # month/day source dates. Text cells retain the original US m/d/Y strings.
    is_excel_date_cell = series.map(
        lambda value: isinstance(value, (pd.Timestamp, datetime, date, np.datetime64))
    )
    parsed = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")

    excel_dates = pd.to_datetime(series.loc[is_excel_date_cell], errors="coerce")
    assert excel_dates.notna().all(), "Unparseable Excel date-formatted cell found."
    assert (excel_dates.dt.day <= 12).all(), (
        "A locale-reversed Excel date cannot be safely repaired because day > 12."
    )
    parsed.loc[is_excel_date_cell] = [
        pd.Timestamp(year=value.year, month=value.day, day=value.month)
        for value in excel_dates
    ]

    text_dates = series.loc[~is_excel_date_cell].astype("string").str.strip()
    parsed.loc[~is_excel_date_cell] = pd.to_datetime(
        text_dates, format="%m/%d/%Y", errors="coerce"
    )
    parse_method = pd.Series(
        np.where(
            is_excel_date_cell,
            "excel_datetime_day_month_reversal",
            "us_string_month_day_year",
        ),
        index=series.index,
        dtype="string",
    )
    return parsed.dt.normalize(), is_excel_date_cell, parse_method

fail_date, is_excel_date_cell, date_parse_method = parse_mixed_fdic_failure_dates(raw["FAILDATE"])
raw_fail_date_text = raw["FAILDATE"].map(raw_date_as_text).astype("string")
source_row_number = pd.Series(raw.index + 2, index=raw.index, name="source_row_number")
assert fail_date.notna().all(), "At least one failure date is missing after parsing."
assert int(is_excel_date_cell.sum()) == EXPECTED_EXCEL_DATE_CELLS
assert int((~is_excel_date_cell).sum()) == EXPECTED_TEXT_FAILURE_DATES

# FDIC_Clean is used only as an independent date-reconciliation control. It is
# never used to populate the handoff.
if "FDIC_Clean" in xls.sheet_names:
    clean_control = pd.read_excel(INPUT_WORKBOOK, sheet_name="FDIC_Clean")
    assert "fail_date" in clean_control.columns
    control_dates = pd.to_datetime(clean_control["fail_date"], errors="coerce").dt.normalize()
    assert len(control_dates) == len(fail_date)
    date_mismatch = fail_date.ne(control_dates)
    assert not date_mismatch.any(), (
        f"{int(date_mismatch.sum())} repaired dates disagree with FDIC_Clean."
    )
else:
    date_mismatch = pd.Series(False, index=raw.index)

date_parsing_summary = pd.DataFrame([
    {"date_source_type": "US text m/d/Y", "rows": int((~is_excel_date_cell).sum()), "mismatches": 0},
    {"date_source_type": "locale-reversed Excel date cell", "rows": int(is_excel_date_cell.sum()), "mismatches": int(date_mismatch.sum())},
    {"date_source_type": "total", "rows": len(raw), "mismatches": int(date_mismatch.sum())},
])
date_reconciliation_audit = pd.DataFrame({
    "source_row_number": source_row_number,
    "cert": pd.to_numeric(raw["CERT"], errors="coerce").astype("Int64"),
    "raw_fail_date_text": raw_fail_date_text,
    "date_parse_method": date_parse_method,
    "fail_date": fail_date,
    "existing_clean_fail_date": control_dates if "FDIC_Clean" in xls.sheet_names else pd.NaT,
    "date_match": ~date_mismatch,
})
date_gate = pd.DataFrame([
    {"check": "all_497_dates_parsed", "passed": fail_date.notna().sum() == 497},
    {"check": "157_excel_dates_repaired", "passed": int(is_excel_date_cell.sum()) == EXPECTED_EXCEL_DATE_CELLS},
    {"check": "340_us_text_dates_parsed", "passed": int((~is_excel_date_cell).sum()) == EXPECTED_TEXT_FAILURE_DATES},
    {"check": "clean_sheet_crosscheck_has_zero_mismatches", "passed": int(date_mismatch.sum()) == 0},
])
date_gate["status"] = np.where(date_gate["passed"], "PASS", "FAIL")
assert date_gate["passed"].all(), date_gate.loc[~date_gate["passed"]]

date_parsing_summary.to_csv(OUTPUT_DIR / "cell_02_date_parse_summary.csv", index=False)
date_reconciliation_audit.to_csv(OUTPUT_DIR / "cell_02_date_crosscheck.csv", index=False)
date_gate.to_csv(OUTPUT_DIR / "cell_02_date_validation_gate.csv", index=False)

print(date_parsing_summary.to_string(index=False))
print("\nCell 02 status: PASS")

## 3. Standardise identifiers, names, locations and financial fields

In [ ]:
# Standardise identifiers, names, locations and financial fields

def normalise_text(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.upper()
    )

cert = pd.to_numeric(raw["CERT"], errors="coerce").astype("Int64")
fin = pd.to_numeric(raw["FIN"], errors="coerce").astype("Int64")
fdic_event_id = pd.to_numeric(raw["ID"], errors="coerce").astype("Int64")
bank_name = normalise_text(raw["NAME"])
location = normalise_text(raw["CITYST"])
location_parts = location.str.rsplit(",", n=1, expand=True)
assert location_parts.shape[1] == 2, "Every location must contain one final city/state separator."
city = location_parts[0].str.strip()
state = location_parts[1].str.strip()

charter_class = normalise_text(raw["CHCLASS1"])
estimated_loss_thousand_usd = pd.to_numeric(raw["COST"], errors="coerce").astype("Float64")
total_assets_thousand_usd = pd.to_numeric(raw["QBFASSET"], errors="coerce").astype("Float64")
total_deposits_thousand_usd = pd.to_numeric(raw["QBFDEP"], errors="coerce").astype("Float64")
resolution = normalise_text(raw["RESTYPE"])
transaction_type = normalise_text(raw["RESTYPE1"])
insurance_fund = normalise_text(raw["SAVR"])

quality_checks = {
    "missing_cert": int(cert.isna().sum()),
    "nonpositive_cert": int((cert <= 0).sum()),
    "missing_fin": int(fin.isna().sum()),
    "missing_fdic_event_id": int(fdic_event_id.isna().sum()),
    "missing_bank_name": int(bank_name.isna().sum()),
    "missing_location": int(location.isna().sum()),
    "invalid_state": int((~state.str.fullmatch(r"[A-Z]{2}", na=False)).sum()),
    "ineligible_charter_class": int((~charter_class.isin(ELIGIBLE_BANK_CLASSES)).sum()),
    "missing_estimated_loss": int(estimated_loss_thousand_usd.isna().sum()),
    "missing_total_assets": int(total_assets_thousand_usd.isna().sum()),
    "missing_total_deposits": int(total_deposits_thousand_usd.isna().sum()),
}
assert quality_checks["missing_cert"] == 0
assert quality_checks["nonpositive_cert"] == 0
assert quality_checks["missing_fin"] == 0
assert quality_checks["missing_fdic_event_id"] == 0
assert quality_checks["missing_bank_name"] == 0
assert quality_checks["missing_location"] == 0
assert quality_checks["invalid_state"] == 0
assert quality_checks["ineligible_charter_class"] == 0
assert quality_checks["missing_estimated_loss"] == 11
assert quality_checks["missing_total_assets"] == 0
assert quality_checks["missing_total_deposits"] == 0

field_quality = pd.DataFrame(
    [{"check": key, "value": value, "status": "PASS"} for key, value in quality_checks.items()]
)
column_mapping = pd.DataFrame({
    "source_column": ["CERT", "CHCLASS1", "CITYST", "COST", "FIN", "ID", "NAME", "QBFASSET", "QBFDEP", "RESTYPE", "RESTYPE1", "SAVR"],
    "canonical_column": ["cert", "charter_class", "location", "estimated_loss_thousand_usd", "fin", "fdic_event_id", "bank_name", "total_assets_thousand_usd", "total_deposits_thousand_usd", "resolution", "transaction_type", "insurance_fund"],
})
standardisation_gate = field_quality.assign(passed=True).loc[:, ["check", "passed", "status"]]
column_mapping.to_csv(OUTPUT_DIR / "cell_03_column_mapping.csv", index=False)
field_quality.to_csv(OUTPUT_DIR / "cell_03_field_quality.csv", index=False)
standardisation_gate.to_csv(OUTPUT_DIR / "cell_03_standardisation_gate.csv", index=False)

source_bank_name = raw["NAME"].astype("string").str.strip()
name_changed = source_bank_name.ne(bank_name)
name_standardisation_audit = pd.DataFrame({
    "source_row_number": raw.index + 2,
    "cert": cert,
    "source_bank_name": source_bank_name,
    "standardised_bank_name": bank_name,
}).loc[name_changed]
name_standardisation_audit.to_csv(
    OUTPUT_DIR / "cell_03_name_standardisation_audit.csv", index=False
)
print(field_quality.to_string(index=False))
print(f"Standardised bank names changed: {len(name_standardisation_audit):,}")
print("\nCell 03 status: PASS")

## 4. Retain genuine failures and preserve excluded assistance events

In [ ]:
# Retain genuine failures and preserve excluded assistance events

events = pd.DataFrame({
    "source_row_number": source_row_number,
    "cert": cert,
    "charter_class": charter_class,
    "bank_name": bank_name,
    "location": location,
    "city": city,
    "state": state,
    "fail_date": fail_date,
    "resolution": resolution,
    "transaction_type": transaction_type,
    "insurance_fund": insurance_fund,
    "estimated_loss_thousand_usd": estimated_loss_thousand_usd,
    "total_assets_thousand_usd": total_assets_thousand_usd,
    "total_deposits_thousand_usd": total_deposits_thousand_usd,
    "fin": fin,
    "fdic_event_id": fdic_event_id,
    "raw_fail_date_text": raw_fail_date_text,
    "date_parse_method": date_parse_method,
})

allowed_resolutions = {"FAILURE", "ASSISTANCE"}
unknown_resolutions = sorted(set(events["resolution"].dropna()) - allowed_resolutions)
assert not unknown_resolutions, f"Unadjudicated resolution values: {unknown_resolutions}"

events["event_classification"] = np.where(
    events["resolution"].eq("FAILURE"),
    "genuine_failure",
    "excluded_non_failure_event",
)
genuine = events.loc[events["resolution"].eq("FAILURE")].copy()
excluded = events.loc[events["resolution"].eq("ASSISTANCE")].copy()
excluded["exclusion_reason"] = "FDIC assistance transaction; not a genuine bank failure"

assert len(genuine) == EXPECTED_GENUINE_FAILURES
assert len(excluded) == EXPECTED_ASSISTANCE_EVENTS
assert len(genuine) + len(excluded) == len(events)

resolution_audit = (
    events.groupby(["resolution", "transaction_type"], dropna=False)
    .size().rename("records").reset_index()
)
resolution_summary = events["resolution"].value_counts(dropna=False).rename_axis("resolution").reset_index(name="observations")
event_gate = pd.DataFrame([
    {"check": "source_contains_487_failures", "passed": len(genuine) == EXPECTED_GENUINE_FAILURES},
    {"check": "source_contains_10_assistance_events", "passed": len(excluded) == EXPECTED_ASSISTANCE_EVENTS},
    {"check": "all_exclusions_have_reason", "passed": excluded["exclusion_reason"].notna().all()},
    {"check": "assistance_fin_equals_zero", "passed": excluded["fin"].eq(0).all()},
    {"check": "included_and_excluded_reconcile_to_497", "passed": len(genuine) + len(excluded) == len(events)},
])
event_gate["status"] = np.where(event_gate["passed"], "PASS", "FAIL")
assert event_gate["passed"].all(), event_gate.loc[~event_gate["passed"]]
resolution_summary.to_csv(OUTPUT_DIR / "cell_04_resolution_summary.csv", index=False)
resolution_audit.to_csv(OUTPUT_DIR / "cell_04_transaction_summary.csv", index=False)
excluded.to_csv(OUTPUT_DIR / "cell_04_excluded_non_failure_events.csv", index=False, date_format="%Y-%m-%d")
event_gate.to_csv(OUTPUT_DIR / "cell_04_event_classification_gate.csv", index=False)

print(resolution_audit.to_string(index=False))
print(f"\nRetained genuine failures: {len(genuine):,}")
print(f"Excluded assistance events: {len(excluded):,}")
print("Cell 04 status: PASS")

## 5. Resolve duplicate conditions and lock one record per failed bank

In [ ]:
# Resolve duplicate conditions and lock one record per failed bank

duplicate_records = genuine.loc[
    genuine.duplicated(keep=False)
    | genuine.duplicated("cert", keep=False)
    | genuine.duplicated(["cert", "fail_date"], keep=False)
    | genuine.duplicated("fdic_event_id", keep=False)
].copy()
duplicate_audit = pd.DataFrame([
    {"duplicate_definition": "CERT", "duplicate_rows": int(genuine.duplicated("cert", keep=False).sum())},
    {"duplicate_definition": "CERT + fail_date", "duplicate_rows": int(genuine.duplicated(["cert", "fail_date"], keep=False).sum())},
    {"duplicate_definition": "FDIC event ID", "duplicate_rows": int(genuine.duplicated("fdic_event_id", keep=False).sum())},
])
duplicate_audit["status"] = np.where(duplicate_audit["duplicate_rows"].eq(0), "PASS", "FAIL")
assert duplicate_audit["duplicate_rows"].eq(0).all(), duplicate_audit
assert genuine["cert"].nunique() == EXPECTED_GENUINE_FAILURES

genuine = genuine.sort_values(["fail_date", "cert", "source_row_number"], kind="stable").drop_duplicates("cert", keep="first").reset_index(drop=True)
duplicate_audit.to_csv(OUTPUT_DIR / "cell_05_duplicate_summary.csv", index=False)
duplicate_records.to_csv(OUTPUT_DIR / "cell_05_duplicate_records.csv", index=False, date_format="%Y-%m-%d")
duplicate_audit.assign(passed=duplicate_audit["duplicate_rows"].eq(0)).to_csv(
    OUTPUT_DIR / "cell_05_duplicate_validation_gate.csv", index=False
)
print(duplicate_audit.to_string(index=False))
print("\nCell 05 status: PASS")

## 6. Validate annual failure counts and outcome-observation coverage

In [ ]:
# Validate annual failure counts and outcome-observation coverage

genuine["failure_year"] = genuine["fail_date"].dt.year.astype("Int64")
genuine["failure_quarter"] = genuine["fail_date"].dt.to_period("Q").astype("string")
outcome_observed_through = pd.Timestamp(OUTCOME_OBSERVED_THROUGH).normalize()
genuine["outcome_observed_through"] = outcome_observed_through

years = pd.Index(range(min(EXPECTED_GENUINE_FAILURE_COUNTS_BY_YEAR), max(EXPECTED_GENUINE_FAILURE_COUNTS_BY_YEAR) + 1), name="year")
observed_by_year = genuine.groupby("failure_year").size().reindex(years, fill_value=0)
expected_by_year = pd.Series(EXPECTED_GENUINE_FAILURE_COUNTS_BY_YEAR, name="expected").reindex(years)
annual_audit = pd.DataFrame({
    "year": years,
    "observed_genuine_failures": observed_by_year.to_numpy(),
    "expected_genuine_failures": expected_by_year.to_numpy(),
})
annual_audit["difference"] = annual_audit["observed_genuine_failures"] - annual_audit["expected_genuine_failures"]
annual_audit["status"] = np.where(annual_audit["difference"].eq(0), "PASS", "FAIL")
assert annual_audit["status"].eq("PASS").all(), annual_audit.loc[annual_audit["status"].eq("FAIL")]
assert int(annual_audit["observed_genuine_failures"].sum()) == EXPECTED_GENUINE_FAILURES

latest_complete_day = outcome_observed_through - pd.Timedelta(days=TARGET_HORIZON_DAYS)
latest_complete_quarter_end = pd.offsets.QuarterEnd(startingMonth=3).rollback(latest_complete_day)
expected_final_test_end = pd.Timestamp(FINAL_TEST_END)

assert genuine["fail_date"].min() == pd.Timestamp(EXPECTED_EARLIEST_FAILURE)
assert genuine["fail_date"].max() == pd.Timestamp(EXPECTED_LATEST_FAILURE)
assert genuine["fail_date"].max() <= outcome_observed_through
assert latest_complete_quarter_end == expected_final_test_end
assert outcome_observed_through >= expected_final_test_end + pd.Timedelta(days=TARGET_HORIZON_DAYS)

temporal_coverage = pd.DataFrame([
    {"metric": "earliest_genuine_failure", "value": genuine["fail_date"].min().date().isoformat()},
    {"metric": "latest_genuine_failure", "value": genuine["fail_date"].max().date().isoformat()},
    {"metric": "outcome_observed_through", "value": outcome_observed_through.date().isoformat()},
    {"metric": "target_horizon_days", "value": TARGET_HORIZON_DAYS},
    {"metric": "latest_complete_calendar_day", "value": latest_complete_day.date().isoformat()},
    {"metric": "latest_eligible_report_quarter_end", "value": latest_complete_quarter_end.date().isoformat()},
])
failure_class_summary = genuine["charter_class"].value_counts().rename_axis("charter_class").reset_index(name="failures")
failure_transaction_summary = genuine["transaction_type"].value_counts().rename_axis("transaction_type").reset_index(name="failures")
temporal_gate = pd.DataFrame([
    {"check": "all_annual_counts_match", "passed": annual_audit["status"].eq("PASS").all()},
    {"check": "annual_counts_sum_to_487", "passed": int(annual_audit["observed_genuine_failures"].sum()) == 487},
    {"check": "latest_eligible_quarter_is_2025Q2", "passed": latest_complete_quarter_end == expected_final_test_end},
])
temporal_gate["status"] = np.where(temporal_gate["passed"], "PASS", "FAIL")
assert temporal_gate["passed"].all(), temporal_gate.loc[~temporal_gate["passed"]]
annual_audit.to_csv(OUTPUT_DIR / "cell_06_annual_failure_counts.csv", index=False)
failure_class_summary.to_csv(OUTPUT_DIR / "cell_06_failure_class_summary.csv", index=False)
failure_transaction_summary.to_csv(OUTPUT_DIR / "cell_06_failure_transaction_summary.csv", index=False)
temporal_coverage.to_csv(OUTPUT_DIR / "cell_06_temporal_summary.csv", index=False)
temporal_gate.to_csv(OUTPUT_DIR / "cell_06_temporal_validation_gate.csv", index=False)

print(annual_audit.to_string(index=False))
print("\n" + temporal_coverage.to_string(index=False))
print("\nCell 06 status: PASS")

## 7. Save and re-read the definitive analysis handoffs

In [ ]:
# Save and re-read the definitive main analysis handoffs

FINAL_COLUMNS = [
    "cert", "fail_date", "bank_name", "city", "state", "charter_class",
    "location", "fdic_event_id", "fin", "resolution", "transaction_type",
    "insurance_fund", "estimated_loss_thousand_usd",
    "total_assets_thousand_usd", "total_deposits_thousand_usd",
    "failure_year", "failure_quarter", "source_row_number",
    "raw_fail_date_text", "date_parse_method", "event_classification",
    "outcome_observed_through",
]
final_failures = genuine.loc[:, FINAL_COLUMNS].copy()
assert final_failures.shape == (EXPECTED_GENUINE_FAILURES, 22)
assert final_failures[["cert", "fail_date", "bank_name"]].notna().all().all()

csv_path = OUTPUT_DIR / "fdic_genuine_bank_failures_2001_2026.csv"
parquet_path = OUTPUT_DIR / "fdic_genuine_bank_failures_2001_2026.parquet"
final_failures.to_csv(csv_path, index=False, date_format="%Y-%m-%d")

parquet_written = False
try:
    final_failures.to_parquet(parquet_path, index=False)
    parquet_written = True
except ImportError as exc:
    if not ALLOW_CSV_ONLY:
        raise RuntimeError(
            "A Parquet engine is required (install pyarrow or fastparquet). "
            "Set NB06_ALLOW_CSV_ONLY=1 only for a non-authoritative development check."
        ) from exc
    print("Development override active: Parquet write skipped because no engine is installed.")

csv_check = pd.read_csv(csv_path, parse_dates=["fail_date", "outcome_observed_through"])
assert csv_check.shape == final_failures.shape
assert csv_check.columns.tolist() == FINAL_COLUMNS
for column in FINAL_COLUMNS:
    left = final_failures[column].reset_index(drop=True)
    right = csv_check[column].reset_index(drop=True)
    value_match = left.eq(right) | (left.isna() & right.isna())
    assert value_match.fillna(False).all(), f"CSV round-trip mismatch in {column}."
if parquet_written:
    parquet_check = pd.read_parquet(parquet_path)
    pd.testing.assert_frame_equal(final_failures, parquet_check, check_dtype=False)

final_gate = pd.DataFrame([
    {"control": "source_rows_equal_497", "passed": len(raw) == EXPECTED_FDIC_SOURCE_ROWS},
    {"control": "genuine_failures_equal_487", "passed": len(final_failures) == EXPECTED_GENUINE_FAILURES},
    {"control": "assistance_events_equal_10", "passed": len(excluded) == EXPECTED_ASSISTANCE_EVENTS},
    {"control": "final_columns_equal_22", "passed": final_failures.shape[1] == 22},
    {"control": "unique_cert_for_each_failure", "passed": final_failures["cert"].nunique() == len(final_failures)},
    {"control": "all_retained_events_are_failures", "passed": final_failures["resolution"].eq("FAILURE").all()},
    {"control": "no_assistance_enters_handoff", "passed": ~final_failures["resolution"].eq("ASSISTANCE").any()},
    {"control": "all_annual_counts_match", "passed": annual_audit["status"].eq("PASS").all()},
    {"control": "outcomes_cover_2025Q2_plus_365_days", "passed": outcome_observed_through >= expected_final_test_end + pd.Timedelta(days=TARGET_HORIZON_DAYS)},
    {"control": "csv_round_trip_exact", "passed": len(csv_check) == len(final_failures) and list(csv_check.columns) == FINAL_COLUMNS},
    {"control": "parquet_written", "passed": parquet_written},
])
authoritative_pass = bool(final_gate["passed"].all())
if not ALLOW_CSV_ONLY:
    assert authoritative_pass, final_gate.loc[~final_gate["passed"]]

final_gate["status"] = np.where(final_gate["passed"], "PASS", "NOT EXECUTED")
final_gate.to_csv(OUTPUT_DIR / "cell_07_final_validation_gate.csv", index=False)

final_summary = pd.DataFrame([
    {"measure": "Source event records", "value": len(raw)},
    {"measure": "Genuine failures retained", "value": len(final_failures)},
    {"measure": "Assistance events excluded", "value": len(excluded)},
    {"measure": "Final handoff columns", "value": final_failures.shape[1]},
    {"measure": "Unique CERT values", "value": final_failures["cert"].nunique()},
    {"measure": "Earliest genuine failure date", "value": final_failures["fail_date"].min()},
    {"measure": "Latest genuine failure date", "value": final_failures["fail_date"].max()},
    {"measure": "Outcome observed through", "value": outcome_observed_through},
    {"measure": "Latest fully eligible report quarter", "value": latest_complete_quarter_end},
    {"measure": "Primary Parquet handoff", "value": parquet_path.name},
    {"measure": "Readable CSV handoff", "value": csv_path.name},
])
final_summary.to_csv(OUTPUT_DIR / "cell_07_final_handoff_summary.csv", index=False, date_format="%Y-%m-%d")

manifest_rows = []
for path, role, sensitivity in [
    (csv_path, "readable genuine-failure handoff", "failure dates; Notebook 07 only"),
    (parquet_path, "primary genuine-failure handoff", "failure dates; Notebook 07 only"),
    (OUTPUT_DIR / "cell_04_excluded_non_failure_events.csv", "assistance exclusion audit", "audit only"),
    (OUTPUT_DIR / "cell_06_annual_failure_counts.csv", "annual-count audit", "audit only"),
    (OUTPUT_DIR / "cell_06_temporal_summary.csv", "outcome-coverage audit", "audit only"),
    (OUTPUT_DIR / "cell_07_final_handoff_summary.csv", "handoff summary", "control"),
    (OUTPUT_DIR / "cell_07_final_validation_gate.csv", "downstream validation gate", "control"),
]:
    if path.exists():
        manifest_rows.append({
            "file": path.name,
            "role": role,
            "leakage_sensitivity": sensitivity,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })
output_manifest = pd.DataFrame(manifest_rows)
output_manifest.to_csv(OUTPUT_DIR / "cell_07_output_manifest.csv", index=False)

print(final_gate.to_string(index=False))
print("\n" + output_manifest.to_string(index=False))
print("\nNotebook 06 authoritative status:", "PASS / LOCKED" if authoritative_pass else "DEVELOPMENT CHECK ONLY")

## Outputs

The main output contains 487 genuine failure records. Assistance events are recorded separately. Notebook 07 uses these records to construct the failure target.
